## Silver Layer Transformation - Branches Standardization & Validation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

In [0]:
# Read bronze table
bronze_df = spark.table("digital_banking.bronze.bronze_branches")

# Display initial count and schema
print(f"Bronze records count: {bronze_df.count()}")
bronze_df.printSchema()

## pyspark.sql.functions.initcap
`pyspark.sql.functions.initcap(col)`
Translate the **first letter** of each word to **upper case** in the sentence.


In [0]:
silver_df = bronze_df.select(
    # Branch Identifiers 
    F.upper(F.trim(F.col("branch_id"))).alias("branch_id"),
    F.upper(F.trim(F.col("branch_code"))).alias("branch_code"),
    F.trim(F.col("branch_name")).alias("branch_name"),
    
    # City Standardization
    F.initcap(F.trim(F.col("city"))).alias("city"),
    
    # State Standardization 
    F.initcap(F.trim(F.col("state"))).alias("state"),
    
    # Region Standardization 
    F.when(
        F.upper(F.trim(F.col("region"))).isin(["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]),
        F.initcap(F.trim(F.col("region")))
    ).otherwise("Unknown").alias("region"),
    
    # Branch Type 
    F.initcap(F.trim(F.col("branch_type"))).alias("branch_type"),
    
    # Manager Name
    F.initcap(F.trim(F.col("manager_name"))).alias("manager_name"),
    
    # Branch Status 
    F.when(
        F.upper(F.trim(F.col("branch_status"))).isin(["ACTIVE", "INACTIVE", "CLOSED", "PENDING"]),
        F.initcap(F.trim(F.col("branch_status")))
    ).otherwise("Unknown").alias("branch_status"),
    
    # Opening Date - Convert to date type
    F.to_date(F.col("opening_date"), "yyyy-MM-dd").alias("opening_date"),
    
    # Add processing metadata
    F.current_timestamp().alias("processed_at"),
    F.lit("Silver").alias("data_layer")
)

In [0]:
# Add data quality validation flags
silver_df = silver_df.withColumns({
    # Validate branch identifiers format
    "is_valid_branch_id": F.when(
        F.col("branch_id").rlike("^BR[0-9]{4}$"),
        F.lit(True)
    ).otherwise(F.lit(False)),
    
    "is_valid_branch_code": F.when(
        F.col("branch_code").rlike("^BNK-[0-9]{4}$"),
        F.lit(True)
    ).otherwise(F.lit(False)),
    
    # Validate location data completeness
    "is_location_complete": F.when(
        F.col("city").isNotNull() & 
        F.col("state").isNotNull() & 
        F.col("region").isNotNull() &
        (F.col("region") != "Unknown"),
        F.lit(True)
    ).otherwise(F.lit(False)),
    
    # Validate status
    "is_valid_status": F.when(
        F.col("branch_status") != "Unknown",
        F.lit(True)
    ).otherwise(F.lit(False)),
    
})

In [0]:
# Show sample of transformed data
display(silver_df.limit(10))

# Region distribution
display(silver_df.groupBy("region").count().orderBy(F.desc("count")))

# Status distribution
display(silver_df.groupBy("branch_status").count().orderBy(F.desc("count")))

In [0]:
# Write to silver table with Delta Lake
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.silver.silver_branches")

print(f"Records written: {silver_df.count()}")